In [2]:
import pandas as pd
import numpy as np

base_file_path = "../data/iterim/"
save_file_path = "../data/preprocessed/"

In [3]:
abstraction_df = pd.read_parquet(f"{base_file_path}abstraction.parquet")
concentration_df = pd.read_parquet(f"{base_file_path}concentration.parquet")
cores_df = pd.read_parquet(f"{base_file_path}cores.parquet")
head_df = pd.read_parquet(f"{base_file_path}head.parquet")
pumping_df = pd.read_parquet(f"{base_file_path}pumping.parquet")
recharge_df = pd.read_parquet(f"{base_file_path}recharge.parquet")
sources_df = pd.read_parquet(f"{base_file_path}sources.parquet")
well_surveys_df = pd.read_parquet(f"{base_file_path}well_surveys.parquet")
wells_df = pd.read_parquet(f"{base_file_path}wells.parquet")
zones_df = pd.read_parquet(f"{base_file_path}zones.parquet")

#### Объединение wells и cores

In [4]:
cores_df = cores_df.dropna(subset=["k_estimate_m_day"])
wells_df = wells_df.rename(columns={"commissioned": "well_commissioned"})


def get_lithology_value(well, z):
    mask = ((cores_df["well_id"] == well) & (cores_df["z_top"] > z) & (cores_df["z_bottom"] < z))

    lithology = cores_df.loc[mask, ["lithology", "k_estimate_m_day", "porosity"]]
    if lithology.empty:
        return np.nan
    else:
        return lithology.iloc[0]


wells_df[["lithology", "k_estimate_m_day", "porosity"]] = wells_df.apply(lambda row: get_lithology_value(row.well_id, row.z), axis=1)
wells_df.loc[wells_df["lithology"].isna(), "lithology"] = "unknown"

#### Объединение wells и zone

In [5]:
def get_zone(x, y):
    mask = (zones_df["x_max"] > x) & (zones_df["y_max"] > y) & (zones_df["x_min"] < x) & (zones_df["y_min"] < y)

    zone_value = zones_df.loc[mask, "zone"]
    if zone_value.empty:
        return np.nan
    else:
        return zone_value.iloc[0]


wells_df["well_zone"] = wells_df.apply(lambda row: get_zone(row.x, row.y), axis=1)

#### Объединение wells и source

In [6]:
def get_distance(source_number, x, y, z):
    z_top =  sources_df.loc[sources_df["source_id"] == f"SRC-0{source_number}", "z_top"]
    z_bottom =  sources_df.loc[sources_df["source_id"] == f"SRC-0{source_number}", "z_bottom"]

    dx = abs(x - sources_df.loc[sources_df["source_id"] == f"SRC-0{source_number}", "x"])
    dy = abs(y - sources_df.loc[sources_df["source_id"] == f"SRC-0{source_number}", "y"])
    dz = abs(z - (z_top + z_bottom) / 2)

    return (np.sqrt(dx ** 2 + dy ** 2 + dz ** 2)).round(3)


wells_df["distance_source_1"] = wells_df.apply(lambda row: get_distance(1, row.x, row.y, row.z), axis=1)
wells_df["distance_source_2"] = wells_df.apply(lambda row: get_distance(2, row.x, row.y, row.z), axis=1)
wells_df["distance_source_3"] = wells_df.apply(lambda row: get_distance(3, row.x, row.y, row.z), axis=1)


source_value = sources_df.loc[sources_df["source_id"] == "SRC-01", "estimated_release_rate_g_day"]


wells_df["src-01_estimated_release_rate_g_day"] = sources_df.loc[sources_df["source_id"] == "SRC-01", "estimated_release_rate_g_day"].item()
wells_df["src-02_estimated_release_rate_g_day"] = sources_df.loc[sources_df["source_id"] == "SRC-02", "estimated_release_rate_g_day"].item()
wells_df["src-03_estimated_release_rate_g_day"] = sources_df.loc[sources_df["source_id"] == "SRC-03", "estimated_release_rate_g_day"].item()


wells_df["src-01_start_date"] = sources_df.loc[sources_df["source_id"] == "SRC-01", "start_date"].item()
wells_df["src-01_end_date"] = sources_df.loc[sources_df["source_id"] == "SRC-01", "end_date"].item()

wells_df["src-02_start_date"] = sources_df.loc[sources_df["source_id"] == "SRC-02", "start_date"].item()
wells_df["src-02_end_date"] = sources_df.loc[sources_df["source_id"] == "SRC-02", "end_date"].item()

wells_df["src-03_start_date"] = sources_df.loc[sources_df["source_id"] == "SRC-03", "start_date"].item()
wells_df["src-03_end_date"] = sources_df.loc[sources_df["source_id"] == "SRC-03", "end_date"].item()


wells_df

,well_id,x,y,z,role,zone,well_commissioned,permeability,lithology,k_estimate_m_day,...,distance_source_3,src-01_estimated_release_rate_g_day,src-02_estimated_release_rate_g_day,src-03_estimated_release_rate_g_day,src-01_start_date,src-01_end_date,src-02_start_date,src-02_end_date,src-03_start_date,src-03_end_date
0,BH-084,569.69,1741.66,-26.98,monitoring,industrial_zone,2018-04-25,63.737530,gravel,56.638,...,1197.936,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
1,BH-271,2199.55,255.93,-11.41,monitoring,central_zone,2018-10-29,26.395163,sand,25.556,...,1261.899,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
2,BH-243,3476.6,2001.54,-10.85,monitoring,water_intake_zone,2018-10-01,26.080097,sand,28.003,...,1844.178,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
3,BH-166,2044.9,1235.17,-20.01,monitoring,central_zone,2018-07-16,16.335054,siltysand,20.354,...,372.21,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
4,BH-167,2447.4,1551.83,-48.09,monitoring,central_zone,2018-07-17,33.444538,siltysand,24.079,...,733.604,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,BH-161,3051.15,1712.12,-17.83,monitoring,water_intake_zone,2018-07-11,26.719461,sand,27.900,...,1356.961,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
315,BH-012,520.86,1699.69,-43.8,monitoring,industrial_zone,2018-02-12,68.331917,gravel,55.455,...,1235.121,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
316,BH-233,3673.47,1638.95,-8.14,monitoring,water_intake_zone,2018-09-21,44.280991,gravel,38.556,...,1960.339,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28
317,BH-139,2573.18,1254.71,-28.88,monitoring,central_zone,2018-06-19,53.428616,gravel,44.547,...,865.116,78.0,31.0,49.0,2019-01-01,NaT,2020-02-01,2024-08-31,2022-05-01,2023-02-28


#### Объединение concentration и head

In [7]:
head = head_df[["well_id", "t", "head"]].rename(
    columns={"t": "head_date", "head": "well_head"}
)
concentration_df = concentration_df.rename(columns={"t": "concentration_t"})
concentration_df = pd.merge_asof(
    concentration_df.sort_values("concentration_t"),
    head.sort_values("head_date"),
    left_on="concentration_t",
    right_on="head_date",
    by="well_id",
    direction="backward",
)

#### Объединение concentration и wells

In [8]:
concentration_df = concentration_df.merge(wells_df[["well_id", "x", "y", "z", "permeability", "zone", "role", "well_commissioned", "lithology", "well_zone"]], on="well_id", how="left", validate="many_to_one")

#### Объединение concentration и recharge

In [9]:
concentration_df["recharge_date"] = (
    concentration_df["concentration_t"].dt.to_period("M").dt.to_timestamp()
)
monthly_recharge = recharge_df[["zone", "t", "recharge"]].rename(
    columns={"t": "recharge_date", "recharge": "zone_recharge"}
)
concentration_df = concentration_df.merge(
    monthly_recharge,
    on=["zone", "recharge_date"],
    how="left",
    validate="many_to_one",
)

#### Объединение concentration и pumping

In [10]:
concentration_df["month"] = concentration_df["concentration_t"].dt.to_period("M").dt.to_timestamp()

pumping_df = pumping_df.rename(columns={"t": "month", "revision": "pumping_revision"})

concentration_df = concentration_df.merge(pumping_df[["well_id", "month", "pumping_rate", "pumping_revision"]], on=["well_id", "month"], how="left", validate="many_to_one")

#### Объединение concentration и abstraction

In [11]:
abstraction_df = abstraction_df.rename(columns={"t": "month", "total_abstraction_m3_day": "abstraction_total_abstraction_m3_day"})

concentration_df = concentration_df.merge(abstraction_df[["month", "abstraction_total_abstraction_m3_day"]], on=["month"], how="left", validate="many_to_one")

In [12]:
display(concentration_df)

,sample_id,well_id,concentration,lod,quality,concentration_t,result_version,unit,lab,head_date,...,role,well_commissioned,lithology,well_zone,recharge_date,zone_recharge,month,pumping_rate,pumping_revision,abstraction_total_abstraction_m3_day
0,LAB-005009,BH-225,0.101,0.1,ok,2020-01-03,1,ug/l,lab-a,NaT,...,monitoring,2018-09-13,siltysand,water_intake_zone,2020-01-01,5.78,2020-01-01,<NA>,NaN,1029.2
1,LAB-000559,BH-061,0.08,0.1,ok,2020-01-03,1,ug/l,lab-b,NaT,...,monitoring,2018-04-02,sand,central_zone,2020-01-01,7.05,2020-01-01,<NA>,NaN,1029.2
2,LAB-001003,BH-025,0.08,0.1,ok,2020-01-03,1,ug/l,lab-b,2020-01-03,...,monitoring,2018-02-25,siltysand,central_zone,2020-01-01,7.05,2020-01-01,<NA>,NaN,1029.2
3,LAB-006686,BH-095,0.08,0.1,ok,2020-01-03,1,ug/l,lab-a,2020-01-03,...,monitoring,2018-05-06,sand,industrial_zone,2020-01-01,4.09,2020-01-01,<NA>,NaN,1029.2
4,LAB-004981,BH-177,0.08,0.1,ok,2020-01-03,1,ug/l,lab-b,2020-01-03,...,monitoring,2018-07-27,sand,water_intake_zone,2020-01-01,5.78,2020-01-01,<NA>,NaN,1029.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6915,LAB-004346,BH-110,0.03,0.05,ok,2025-12-24,1,ug/l,lab-a,2025-12-24,...,monitoring,2018-05-21,sand,industrial_zone,2025-12-01,5.89,2025-12-01,<NA>,NaN,1214.5
6916,LAB-001594,BH-212,3.523,0.05,ok,2025-12-24,1,ug/l,lab-b,2025-12-24,...,monitoring,2018-08-31,sand,water_intake_zone,2025-12-01,8.32,2025-12-01,<NA>,NaN,1214.5
6917,LAB-003127,BH-008,0.03,0.05,ok,2025-12-25,1,ug/l,lab-b,2025-12-25,...,monitoring,2018-02-08,gravel,industrial_zone,2025-12-01,5.89,2025-12-01,<NA>,NaN,1214.5
6918,LAB-000977,BH-308,0.03,0.05,ok,2025-12-25,1,ug/l,lab-a,2025-08-14,...,monitoring,2018-12-05,sand,industrial_zone,2025-12-01,5.89,2025-12-01,<NA>,NaN,1214.5


In [13]:
concentration_df.to_parquet(f"{save_file_path}merged_data.parquet", engine="pyarrow")